<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type: Ranking & Scoring:**

I am framing this as a Search Opportunity Scoring and Ranking problem.
A standard binary classifier ("needs update: yes/no") isn't practical here. Our content team only has bandwidth to rewrite about 20 to 50 articles a week. If a classifier flags 3,000 pages, the team still doesn't know which 20 to pick first.
Instead, we need a continuous score (0–100) to output a sorted priority list. That way, editors can just take the top 50 rows off the queue every Monday, focusing on pages with decent search volume where traffic loss actually hurts revenue.

In [1]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

In [2]:

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [3]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.head())


             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76 

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy Definition:**
Since Google Search Console doesn't give us a clear "this page needs an update" column, we have to build a proxy target. True ground truth would mean running A/B tests on rewritten content to see if traffic recovers, which we can't do upfront.
My proxy target is is_declining_label (defined as trend_direction == 'down').
It's not a perfect label, but a page suffering a sustained 30-day drop in organic clicks is our best indicator that an article is decaying and needs an editorial look.

*Limitation of Proxy:*  Using trend_direction == 'down' as a target assumes all traffic loss is content degradation. It may misclassify seasonal keywords or intent shifts as content decay

In [4]:
#  View a few rows of the dataset
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Check label distribution
decline_counts = df["is_declining_label"].value_counts()
decline_rate = df["is_declining_label"].mean()

print(f"Declining pages (Label 1): {decline_counts[1]:,} rows")
print(f"Stable/Growing pages (Label 0): {decline_counts[0]:,} rows")
print(f"Baseline decline rate in dataset: {decline_rate * 100:.1f}%")

Declining pages (Label 1): 16,262 rows
Stable/Growing pages (Label 0): 13,738 rows
Baseline decline rate in dataset: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Success Metric: Precision@50**



*Why Precision@50?*

My primary metric is Precision@50.

The cost of misclassification is completely lopsided here:

False Positives are expensive: Recommending a healthy page wastes 3 or 4 hours of an editor's time auditing something that didn't need fixing.

False Negatives are cheap: Missing a declining page just means it stays in the backlog for another week until the next run.

Since editors can only take on ~50 articles a week, I only care about how accurate our top 50 picks are. If 40 out of our top 50 recommendations are actually declining pages, that's an 80% Precision@50—a huge step up over our baseline rules.

In [5]:
# Naive Baseline 1: Rank purely by raw Search Volume
top_50_volume = df.sort_values("search_volume", ascending=False).head(50)
p50_volume = top_50_volume["is_declining_label"].mean()

# Naive Baseline 2: Rank purely by lowest Click-Through Rate (CTR)
top_50_ctr = df.sort_values("ctr", ascending=True).head(50)
p50_ctr = top_50_ctr["is_declining_label"].mean()

print(f"Naive Rule 1 (Search Volume Alone) Precision@50: {p50_volume * 100:.1f}%")
print(f"Naive Rule 2 (Lowest CTR Alone) Precision@50:      {p50_ctr * 100:.1f}%")
print(f"Random Baseline (Dataset Average):                {decline_rate * 100:.1f}%")

Naive Rule 1 (Search Volume Alone) Precision@50: 42.0%
Naive Rule 2 (Lowest CTR Alone) Precision@50:      50.0%
Random Baseline (Dataset Average):                54.2%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:**

One row = One unique published article (content_id).

Each observation blends 90-day Search Console stats (clicks, impressions, position, CTR) with site engagement numbers (scroll rate, sessions) and article metadata (word count, content age).

In [6]:
# Inspect the Unit of Analysis Dataframe
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Display key unit identifier columns & metrics
unit_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "content_age_days", "days_since_last_update", "is_declining_label"
]

print("\nSample Rows (Unit of Analysis):")
df[unit_cols].head(3)

Dataset Shape: 30,000 rows × 45 columns

Sample Rows (Unit of Analysis):


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,10.6,0.76,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,20.3,0.05,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,36.5,0.09,141,20,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Simple rule-of-thumb logic sounds good on paper, but it breaks down on real search data.

When I tested a basic rule—"flag any page older than 180 days"—it only hit 48.3% precision, which is actually worse than random guessing in this dataset (54.2%).

That happens because search metrics don't follow clean linear rules:

Many 2-year-old evergreen guides stay locked at #1 on Google without any maintenance.

A 2% CTR on Position #2 is alarming, but a 2% CTR on Position #35 is completely normal.

A static IF statement can't naturally weigh position, impressions, and CTR drops together. That non-linear boundary is where a model actually adds value.

In [7]:
# # Check how the simple age rule performs: Flag pages where content_age_days > 180
stale_rule_mask = df["content_age_days"] > 180
stale_pages = df[stale_rule_mask]

# Calculate Precision & Recall of the simple rule
stale_precision = stale_pages["is_declining_label"].mean() if len(stale_pages) > 0 else 0
stale_count = len(stale_pages)

print(f"Pages flagged by 'Age > 180 days' rule: {stale_count:,}")
print(f"Precision of simple rule: {stale_precision * 100:.1f}%")
print(f"Overall dataset decline rate: {decline_rate * 100:.1f}%")

Pages flagged by 'Age > 180 days' rule: 17,728
Precision of simple rule: 48.3%
Overall dataset decline rate: 54.2%


These simple ranking rules don't perform very well, which supports using multiple features instead of relying on one metric.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check & Limitations

Honestly, the weakest part of this framing is the target proxy itself. I'm using `trend_direction == 'down'` as a stand-in for "needs a refresh," but those aren't the same thing:

* **Confounding Variables:** A page might lose traffic due to a broad Google algorithm update or internal keyword cannibalization (another client page ranking for the same query). A content rewrite won't fix either issue, but the model will still score it high.
* **Seasonality:** Seasonal content (e.g., "best gifts 2025") naturally trends down every January. That's a calendar shift, not a content quality decay. In a production pipeline, I'd want to filter out or flag seasonal pages before queueing them for editors.
* **Proxy vs. Real Impact:** The real test of whether Precision@50 matters isn't in this notebook—it's whether the articles we refresh actually regain traffic 30–60 days later. For now, treating this proxy as a first pass lets us build a working v1 pipeline, but post-refresh traffic recovery is the true metric to track long-term.

Interestingly, ranking only by search volume performed worse than the overall dataset average. This suggests that search volume alone is not enough to identify pages that need refreshing.